# NB2 · Xu hướng & đồng ngành → Phát hiện tín hiệu → Chấm điểm & xếp hạng (Bước 4–6)
**FinReport Intelligence · Festival Chuyển đổi số 2026 · Bảng C · Case 3**

| | |
|---|---|
| **Đọc từ** | `03_outputs/step0/master_panel.csv` · `03_outputs/step1_3/metrics_wide.csv`, `metrics_supplementary.csv` · nhãn mẫu `financial_scenarios_sample.csv` (chỉ để đánh giá) |
| **Ghi ra** | `03_outputs/step4_6/` |
| **Thư viện** | `lib/finreport_signals.py` (+ `finreport_common`, `finreport_metrics`) |

| Bước | Câu hỏi | Cách làm |
|---|---|---|
| **4. Xu hướng & đồng ngành** | DN đang tốt lên hay xấu đi? So với ngành ra sao? | So với năm gốc, trung bình 2 năm gần nhất, median/P25/P75 ngành theo từng năm |
| **5. Phát hiện tín hiệu** | Có dấu hiệu gì đáng chú ý? | Tín hiệu chỉ bật khi **mức + xu hướng + đồng ngành** cùng xác nhận; chọn kịch bản theo **nguyên nhân gốc** |
| **6. Chấm điểm & xếp hạng** | Mức độ nào? Xem DN nào trước? | `risk_score` 6 thành phần, severity, health, priority_rank, confidence, lý do và bằng chứng |

> Cảnh báo là **tín hiệu hỗ trợ phân tích** – không phải kết luận gian lận, kết luận kiểm toán hay khuyến nghị đầu tư.

## 0. Kết nối Drive, nạp thư viện, kiểm tra kết quả NB1

In [ ]:
import os, sys, importlib, time
try:
    from google.colab import drive
    drive.mount("/content/drive")
    PROJECT = "/content/drive/MyDrive/FinReport_Intelligence"
except ImportError:
    PROJECT = os.environ.get("FINREPORT_PROJECT", "FinReport_Intelligence")

sys.path.insert(0, PROJECT + "/lib")
import finreport_common, finreport_metrics, finreport_signals
for mod in (finreport_common, finreport_metrics, finreport_signals):
    importlib.reload(mod)
from finreport_common import *
import finreport_metrics as fm
import finreport_signals as fs
import pandas as pd, numpy as np
pd.set_option("display.max_colwidth", 140); pd.set_option("display.width", 240); pd.set_option("display.max_columns", 40)

cfg = setup(PROJECT)
STEP = "step4_6"

In [ ]:
master = read_output("step0", "master_panel.csv")
metrics_wide = read_output("step1_3", "metrics_wide.csv")
supp = read_output("step1_3", "metrics_supplementary.csv")

audit = read_audit()
s3 = audit[audit["step"] == "step3_verify_qa"].tail(1)
assert len(s3), "Chưa có kết quả NB1 – hãy chạy NB1_Calculate trước"
PARENT_RUN = s3.iloc[0]["run_id"]
print(f"NB1 gần nhất: {PARENT_RUN} · {s3.iloc[0]['verdict']}")
print(f"master_panel {master.shape} · metrics_wide {metrics_wide.shape} · supplementary {supp.shape}")

labels_path, _ = locate_pack_file("financial_scenarios_sample")
labels = pd.read_csv(labels_path, encoding="utf-8-sig", keep_default_na=False) if labels_path is not None else None
print("Nhãn mẫu:", "có" if labels is not None else "CHƯA CÓ", "" if labels is None else f"({len(labels)} company-year)")

---
## BƯỚC 4 · Xu hướng nhiều năm & so sánh đồng ngành

Mỗi company-year được bổ sung 3 loại đặc trưng (chỉ dùng dữ liệu **đến năm đó**, không nhìn tương lai):

| Loại | Ví dụ | Ý nghĩa |
|---|---|---|
| **So với năm gốc** của chính DN | `d_gm` (biên gộp thay đổi bao nhiêu điểm %), `dso_rel` (DSO gấp bao nhiêu lần năm đầu), `d_de`, `icr_drop` | Phát hiện **xu hướng xấu đi/tốt lên** |
| **Trung bình 2 năm gần nhất** | `ar_gap2` = tăng trưởng phải thu − tăng trưởng doanh thu; `inv_gap2`; `debt_g_min2` | Lọc nhiễu 1 năm, đòi hỏi tín hiệu **kéo dài** |
| **Đồng ngành theo từng năm** (benchmark động) | median, P25, P75 của 12 DN; `dso_vs_peer`, `invd_vs_peer` | Phân biệt "cả ngành cùng thế" với "riêng DN này bất thường" |

In [ ]:
t0 = time.perf_counter()
RUN_S4 = new_run_id("S4")
features = fs.build_features(master, metrics_wide, supp)
ms_s4 = (time.perf_counter() - t0) * 1000
print(f"Run {RUN_S4}: {features.shape[0]} dòng × {features.shape[1]} đặc trưng · {ms_s4:.0f} ms")

# Bảng benchmark ngành theo năm
peer = (features.groupby("fiscal_year")[["revenue_growth_pct", "gross_margin_pct", "operating_margin_pct", "current_ratio_x",
                                         "debt_to_equity_x", "interest_coverage_x", "dso_days", "inventory_days"]]
        .agg(["median"]).round(2))
peer.columns = [c[0] for c in peer.columns]
print("\nMedian ngành theo năm:")
display(peer)

In [ ]:
# Ví dụ xu hướng của một DN so với ngành (đổi mã tùy ý)
CID = "C003"
cols = ["fiscal_year", "dso_days", "dso_rel", "peer_dso_med", "dso_vs_peer", "ar_gap2",
        "cfo_to_net_income_x", "gross_margin_pct", "d_gm", "debt_to_equity_x", "d_de"]
display(features[features["company_id"] == CID][cols].round(2))

In [ ]:
save_output(features, STEP, "features.csv")
save_output(peer.reset_index(), STEP, "peer_benchmarks.csv")
append_audit({"run_id": RUN_S4, "step": "step4_trend_peer", "parent_run": PARENT_RUN, "verdict": "PASS",
              "features": int(features.shape[1]), "processing_time_ms": round(ms_s4, 1),
              "outputs": [f"03_outputs/{STEP}/features.csv", f"03_outputs/{STEP}/peer_benchmarks.csv"]})

---
## BƯỚC 5 · Phát hiện tín hiệu

**Ngưỡng** – tất cả nằm trong một bảng duy nhất (`fs.THRESHOLDS`), là các mốc tròn theo thông lệ phân tích tài chính:

In [ ]:
display(pd.Series(fs.THRESHOLDS, name="giá trị").to_frame())

**Tín hiệu** (mỗi tín hiệu = điều kiện kết hợp nhiều chiều):

| Tín hiệu | Điều kiện |
|---|---|
| `S_MARGIN` | Biên gộp giảm ≥ 3 điểm % so với năm gốc |
| `S_REC_P` / `S_REC` | Phải thu tăng nhanh hơn doanh thu ≥ 15đ%/năm (TB 2 năm) / **và** DSO ≥ 1,25× năm gốc |
| `S_INV_P` / `S_INV` | Tồn kho tăng nhanh hơn giá vốn ≥ 10đ%/năm / **và** ngày tồn kho ≥ 1,25× năm gốc |
| `S_LEV` | Nợ vay tăng ≥ 20%/năm **hai năm liền** **và** D/E tăng ≥ 0,10 |
| `S_CASH` | CFO/LNST < 0,8 (khi LNST > 0) hoặc CFO âm |
| `S_LIQ` | CFO/LNST ≤ 0,3 hoặc CFO âm, **và** ≥ 45% nợ vay đáo hạn trong 12 tháng |
| `S_DISTRESS` / `S_RECOV` | Biên hoạt động < 2% / từng khủng hoảng, nay có lãi và biên gộp hồi ≥ 3đ% từ đáy |
| `S_STRONG` / `S_IMPROVE` | Top ngành (biên, ICR ≥ 20, D/E ≤ P25, thanh khoản ≥ P75) / biên cải thiện + giảm nợ + vốn lưu động tốt lên |

**Kịch bản chính** được chọn theo thứ tự **nguyên nhân gốc**: Khủng hoảng → Phục hồi → Vượt trội → Đa yếu tố → Thanh khoản →
Lợi nhuận–dòng tiền phân kỳ → Tăng trưởng nhưng dòng tiền yếu → Phải thu → Tồn kho → Mở rộng nợ → Suy giảm biên → Cải thiện → Bình thường.
Tên tag phụ thuộc kịch bản (ví dụ dòng tiền yếu do phải thu gọi là `CFO_PRESSURE`, do tăng trưởng nóng gọi là `CFO_WEAKNESS`) – đúng từ vựng nhãn BTC.

In [ ]:
t0 = time.perf_counter()
RUN_S5 = new_run_id("S5")
signals = fs.detect_signals(features)
ms_s5 = (time.perf_counter() - t0) * 1000
sig_cols = [c for c in signals.columns if c.startswith("S_")]
print(f"Run {RUN_S5}: {len(sig_cols)} tín hiệu × {len(signals)} dòng · {ms_s5:.0f} ms")
print("\nSố company-year bật từng tín hiệu:")
display(signals[sig_cols].sum().to_frame("số dòng bật").T)

In [ ]:
# Tín hiệu đang bật của từng DN qua các năm
on = signals.melt(id_vars=fm.KEY, value_vars=sig_cols)
on = on[on["value"]].groupby(fm.KEY)["variable"].apply(lambda s: " ".join(x[2:] for x in s)).unstack("fiscal_year").fillna("")
display(on)
save_output(signals, STEP, "signals.csv")
append_audit({"run_id": RUN_S5, "step": "step5_signals", "parent_run": RUN_S4, "verdict": "PASS",
              "thresholds": fs.THRESHOLDS, "signals_on": {k: int(v) for k, v in signals[sig_cols].sum().items()},
              "processing_time_ms": round(ms_s5, 1), "outputs": [f"03_outputs/{STEP}/signals.csv"]})

---
## BƯỚC 6 · Chấm điểm, mức độ, sức khỏe, xếp hạng ưu tiên

**`risk_score` ∈ [0, 1]** – tổng có trọng số 6 thành phần (mỗi thành phần ∈ [0, 1]):

| Thành phần | Trọng số | Cách tính |
|---|---:|---|
| Dòng tiền (`cash`) | 0,20 | 1 − CFO/LNST (CFO âm → 1) |
| Khả năng trả lãi (`coverage`) | 0,20 | (8 − ICR) / 8 |
| Đòn bẩy (`leverage`) | 0,15 | ½·D/E + ½·(mức tăng D/E / 0,4) |
| Biên lợi nhuận (`margin`) | 0,15 | ½·(giảm biên gộp/10) + ½·(giảm biên HĐ/10) |
| Vốn lưu động (`working_capital`) | 0,15 | max(DSO, ngày tồn kho so với gốc − 1) |
| Lỗ (`loss`) | 0,15 | 1 nếu LNST ≤ 0 |

**Mức độ**: kịch bản tích cực → LOW · phục hồi → LOW/MEDIUM · rủi ro: **CRITICAL** nếu ICR < 3 hoặc lỗ; **HIGH** nếu ≥ 2 tag rủi ro; **MEDIUM** nếu 1.
**Sức khỏe**: HIGH_RISK nếu CRITICAL, biên hoạt động giảm ≥ 3đ%, đa yếu tố, hoặc CFO/doanh thu < 2%; còn lại WATCH.
**Xếp hạng**: mức độ trước, rồi `risk_score` (1 = xem trước nhất). **Độ tin cậy**: đủ lịch sử (2021 thấp hơn) × không sát ngưỡng.

In [ ]:
t0 = time.perf_counter()
RUN_S6 = new_run_id("S6")
pred = fs.score(features, signals)
pred["confidence"] = fs.confidence(features)
pred["alert_reason"], pred["evidence_fields"] = fs.explain(features, pred)
ms_s6 = (time.perf_counter() - t0) * 1000
print(f"Run {RUN_S6}: {len(pred)} dòng · {ms_s6:.0f} ms")

print("\nKịch bản theo DN × năm:")
display(pred.pivot(index="company_id", columns="fiscal_year", values="predicted_primary_scenario"))
print("Mức độ theo DN × năm:")
display(pred.pivot(index="company_id", columns="fiscal_year", values="predicted_severity"))

In [ ]:
print("Risk score theo DN × năm:")
display(pred.pivot(index="company_id", columns="fiscal_year", values="risk_score").round(2))
print("\nDanh sách ưu tiên – 12 company-year cần xem trước:")
display(pred.sort_values("priority_rank").head(12)[["priority_rank", "company_id", "fiscal_year", "predicted_severity",
                                                    "predicted_primary_scenario", "risk_score", "confidence"]])

In [ ]:
# Giải thích & truy vết của một vài cảnh báo
for cid, y in [("C003", 2024), ("C007", 2025), ("C012", 2025), ("C009", 2024)]:
    r = pred[(pred["company_id"] == cid) & (pred["fiscal_year"] == y)].iloc[0]
    print(f"■ {cid}/{y} · {r.predicted_severity} · {r.predicted_overall_health} · risk {r.risk_score:.2f} · conf {r.confidence:.2f}")
    print("  Tags   :", r.alert_tags or "(không)")
    print("  Lý do  :", r.alert_reason)
    print("  Bằng chứng:", r.evidence_fields.replace("|", "\n              "))
    print()

---
## Đánh giá với nhãn mẫu BTC

So 18 company-year có nhãn trong `financial_scenarios_sample.csv`. Đối chứng với **baseline**: luật 1 năm, 1 chỉ tiêu, ngưỡng cố định, không xu hướng, không ngữ cảnh.

> ⚠ **Lưu ý trung thực**: luật được thiết kế khi đã quan sát 18 nhãn này → kết quả là **trong mẫu** (in-sample), có thể lạc quan.
> Để kiểm tra khả năng tổng quát, notebook chạy thêm: (1) độ nhạy ngưỡng, (2) độ nhất quán theo thời gian trên 42 dòng **không có nhãn**.

In [ ]:
if labels is not None:
    ev_rules = fs.evaluate(pred, labels)
    ev_base = fs.evaluate(fs.baseline(features), labels)
    comp = pd.DataFrame({"Luật đa tín hiệu (hệ thống)": ev_rules, "Baseline 1 chỉ tiêu": ev_base}).drop("n_labels")
    comp["chênh lệch"] = comp.iloc[:, 0] - comp.iloc[:, 1]
    display(comp.astype(float).round(3))

In [ ]:
if labels is not None:
    rows = fs.compare_rows(pred, labels)
    display(rows[["company_id", "fiscal_year", "primary_scenario", "predicted_primary_scenario", "severity",
                  "predicted_severity", "overall_health", "predicted_overall_health", "tags_missing", "tags_extra"]])

### Độ nhạy ngưỡng
(1) Nhân **đồng loạt** mọi ngưỡng ×0,8 → ×1,2. (2) Đổi **từng** ngưỡng ±20%, giữ nguyên các ngưỡng khác.
Nếu kết quả chỉ đúng ở đúng bộ ngưỡng hiện tại → dấu hiệu "khớp quá mức" (overfit). Nếu ổn định → luật bền vững.

In [ ]:
if labels is not None:
    sens = fs.sensitivity(master, metrics_wide, supp, labels)
    display(sens[["threshold_factor", "tag_f1_micro", "scenario_accuracy", "severity_accuracy", "health_accuracy", "high_critical_recall"]])
    oat = fs.one_at_a_time(master, metrics_wide, supp, labels)
    print(f"Đổi từng ngưỡng ±20%: {(oat['Δf1'] == 0).mean():.0%} trường hợp không làm thay đổi F1 tag.")
    print("Các ngưỡng nhạy nhất:")
    display(oat.sort_values("Δf1").head(6))

### Nhất quán theo thời gian trên dòng không có nhãn
Một hệ thống "học thuộc" 18 nhãn sẽ cho kết quả lộn xộn ở các năm còn lại. Kiểm tra: mỗi DN có giữ kịch bản ổn định không,
và mức độ có diễn biến hợp lý (xấu dần hoặc phục hồi dần) không.

In [ ]:
unl = pred if labels is None else pred.merge(labels[fm.KEY], on=fm.KEY, how="left", indicator=True).query("_merge == 'left_only'")
stab = (pred[pred["fiscal_year"] >= 2023].groupby("company_id")["predicted_primary_scenario"].nunique() == 1).mean()
sev_num = pred.assign(s=pred["predicted_severity"].map(fs.SEV_ORDER)).pivot(index="company_id", columns="fiscal_year", values="s")
jumps = (sev_num.diff(axis=1).abs() >= 2).sum().sum()
print(f"Dòng không có nhãn: {len(unl)}")
print(f"DN giữ nguyên kịch bản giai đoạn 2023–2025: {stab:.0%}")
print(f"Số lần mức độ nhảy ≥ 2 bậc giữa hai năm liên tiếp: {int(jumps)}")

---
## Lưu kết quả & báo cáo

In [ ]:
save_output(pred, STEP, "predictions.csv")
outs = ["predictions.csv"]
if labels is not None:
    save_output(rows, STEP, "label_comparison.csv")
    save_output(comp.reset_index().rename(columns={"index": "metric"}), STEP, "evaluation_vs_baseline.csv")
    save_output(sens, STEP, "sensitivity_global.csv")
    save_output(oat, STEP, "sensitivity_one_at_a_time.csv")
    outs += ["label_comparison.csv", "evaluation_vs_baseline.csv", "sensitivity_global.csv", "sensitivity_one_at_a_time.csv"]

KPI = {} if labels is None else {k: round(float(v), 4) for k, v in ev_rules.items() if k != "n_labels"}
KPI.update({"baseline_tag_f1": None if labels is None else round(float(ev_base["tag_f1_micro"]), 4),
            "scenario_stability_2023_2025": round(float(stab), 4), "severity_jumps_ge2": int(jumps)})
append_audit({"run_id": RUN_S6, "step": "step6_scoring", "parent_run": RUN_S5, "verdict": "PASS",
              "model_name": fs.MODEL_NAME, "risk_weights": fs.RISK_WEIGHTS, "kpi": KPI, "evaluation": "in-sample (18 nhãn mẫu)",
              "processing_time_ms": round(ms_s6, 1), "outputs": [f"03_outputs/{STEP}/{o}" for o in outs]})
pd.Series(KPI).to_frame("giá trị")

In [ ]:
pct = lambda x: f"{x:.1%}"
L = ["# Báo cáo NB2 – Xu hướng, đồng ngành, tín hiệu, chấm điểm", "",
     f"- **Thời điểm:** {now_iso()} · **Mô hình:** {fs.MODEL_NAME}",
     f"- **Chuỗi lần chạy:** `{PARENT_RUN}` → `{RUN_S4}` → `{RUN_S5}` → `{RUN_S6}`", ""]
if labels is not None:
    L += ["## Đánh giá trên 18 nhãn mẫu (in-sample)", "", "| Chỉ số | Hệ thống | Baseline |", "|---|---:|---:|"]
    names = {"tag_precision": "Tag Precision", "tag_recall": "Tag Recall", "tag_f1_micro": "Tag F1 (micro)",
             "tag_f1_macro": "Tag F1 (macro)", "scenario_accuracy": "Kịch bản – accuracy",
             "scenario_macro_recall": "Kịch bản – macro recall", "severity_accuracy": "Mức độ – accuracy",
             "health_accuracy": "Sức khỏe – accuracy", "high_critical_recall": "Recall HIGH/CRITICAL",
             "false_alarm_rate_on_low": "Báo động nhầm trên DN LOW"}
    L += [f"| {v} | {pct(ev_rules[k])} | {pct(ev_base[k])} |" for k, v in names.items()]
    L += [f"| Spearman risk_score ~ severity | {ev_rules['spearman_risk_vs_severity']:.2f} | – |", "",
          "## Độ bền", "",
          f"- Nhân đồng loạt ngưỡng ×0,8…×1,2: Tag F1 trong khoảng {sens['tag_f1_micro'].min():.2f}–{sens['tag_f1_micro'].max():.2f}.",
          f"- Đổi từng ngưỡng ±20%: {(oat['Δf1'] == 0).mean():.0%} trường hợp không đổi F1.",
          f"- Nhất quán theo thời gian (42 dòng không nhãn): {pct(stab)} DN giữ kịch bản 2023–2025; {int(jumps)} lần mức độ nhảy ≥ 2 bậc.", "",
          "## Sai khác với nhãn", ""]
    diff = rows[(~rows["scenario_ok"]) | (~rows["severity_ok"]) | (~rows["health_ok"]) | (rows["tags_missing"] != "") | (rows["tags_extra"] != "")]
    L += ["Không có."] if diff.empty else [f"- {r.company_id}/{r.fiscal_year}: thiếu `{r.tags_missing}` thừa `{r.tags_extra}`" for r in diff.itertuples()]
L += ["", "## Danh sách ưu tiên (top 12)", "", "| # | DN | Năm | Mức độ | Kịch bản | Risk | Lý do |", "|---:|---|---|---|---|---:|---|"]
L += [f"| {r.priority_rank} | {r.company_id} | {r.fiscal_year} | {r.predicted_severity} | {r.predicted_primary_scenario} | {r.risk_score:.2f} | {r.alert_reason[:110]}… |"
      for r in pred.sort_values("priority_rank").head(12).itertuples()]
L += ["", "> Cảnh báo là tín hiệu hỗ trợ phân tích – không phải kết luận gian lận, kiểm toán hay khuyến nghị đầu tư."]
rep = cfg.step_dir(STEP) / "step4_6_report.md"
rep.write_text("\n".join(L), encoding="utf-8")
print("Đã ghi:", rep)
for f_ in sorted(cfg.step_dir(STEP).iterdir()):
    print("  ", f_.name)